<a href="https://colab.research.google.com/github/SravaniPurra/Poject-of-AI-ML/blob/main/3_support_assistant_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q sentence-transformers chromadb langgraph fastapi uvicorn pydantic

In [3]:
import os
os.makedirs("docs", exist_ok=True)
print("docs folder created")

docs folder created


In [4]:
documents = {
    "doc_01.txt": """Zepto delivers grocery and household essentials to serviceable pin codes within 10 to 30 minutes of order confirmation, depending on the customer's delivery zone and current order volume. Standard delivery is free on orders over INR 149; orders below this threshold incur a flat INR 25 delivery fee. Priority delivery, which reserves the next available rider slot, is available at checkout for an additional INR 15. Zepto does not currently deliver to addresses outside its listed serviceable pin codes.""",

    "doc_02.txt": """Grocery and perishable items may be reported for a return within 24 hours of delivery if damaged, spoiled, or incorrect; non-perishable packaged items may be returned within 7 days of delivery in unopened, resalable condition. Approved refunds are credited to the original payment method within 3–5 business days, or instantly to the Zepto wallet if the customer opts for wallet credit. Personal care items that have been opened are non-returnable except in the case of a manufacturing defect. Return pickup, where required, is arranged free of cost by Zepto.""",

    "doc_03.txt": """Zepto offers three account tiers: Basic (free, default tier, standard delivery fees apply), Zepto Pass (INR 49 per month, free standard delivery on all orders and 5% off select categories), and Zepto Pass+ (INR 99 per month, free priority delivery, 10% off select categories, and early access to limited-time deals 24 hours before they go live to Basic and Pass members). Membership can be cancelled at any time from account settings; cancelling stops the next billing cycle but does not refund the current membership period.""",

    "doc_04.txt": """Every Zepto order shows a live rider-tracking map from the moment it is packed until delivery, accessible from the 'Track Order' screen. Estimated delivery time updates automatically as the rider moves. If an order's status shows no movement for more than 20 minutes past its original estimated delivery time, customers should contact support directly rather than continue waiting, since this indicates a likely delivery issue.""",

    "doc_05.txt": """Orders can be cancelled free of cost any time before the order status changes to 'Packed', typically within the first 2 minutes of placing the order. Once an order has been packed, it can no longer be cancelled through the app, since the rider is dispatched immediately after packing given Zepto's quick-delivery model. If a packed order cannot be delivered due to a Zepto-side issue (for example, rider unavailability), the order is auto-cancelled and fully refunded without any cancellation fee.""",

    "doc_06.txt": """If an order arrives with damaged, spoiled, or missing items, customers must report it within 24 hours of delivery through the 'Report an Issue' button on the order page. Zepto ships a free replacement or issues a full refund for damaged, spoiled, or missing items without requiring the customer to return the original item, unless the order value exceeds INR 1000, in which case a photo of the issue must be submitted through the report form before a replacement or refund is processed.""",

    "doc_07.txt": """Zepto gift cards are available in fixed denominations of INR 100, INR 250, INR 500, and INR 1000, and are delivered by email or SMS within minutes of purchase. Gift cards are valid for 1 year from the date of issue and carry no maintenance fees. Gift card balance can be combined with one other payment method at checkout but cannot be combined with another gift card in the same transaction. Gift card balance cannot be redeemed for cash except where required by law.""",

    "doc_08.txt": """Zepto customer support is available via in-app chat 24 hours a day, 7 days a week, given the time-sensitive nature of quick commerce deliveries. Average in-app chat response time is under 2 minutes. Email support is also available for non-urgent queries and is answered within 24 hours on business days. Phone support is not offered."""
}

for filename, text in documents.items():
    with open("docs/" + filename, "w", encoding="utf-8") as file:
        file.write(text)

print("All 8 documents created successfully!")

All 8 documents created successfully!


In [5]:
import os
print(os.listdir("docs"))

['doc_02.txt', 'doc_03.txt', 'doc_04.txt', 'doc_08.txt', 'doc_07.txt', 'doc_05.txt', 'doc_06.txt', 'doc_01.txt']


In [6]:
documents = []
for i in range(1, 9):
  file_name = f"docs/doc_{i:02d}.txt"
  with open(file_name, "r", encoding="utf-8") as file:
    text = file.read()
  documents.append({
     "id": f"doc_{i:02d}",
    "text": text
    })
print("Number of documents:", len(documents))

Number of documents: 8


In [7]:
chunks = []
for doc in documents:
  chunks.append({
        "id": doc["id"] + "_chunk_01",
        "text": doc["text"],
        "document_id": doc["id"]
    })

print("Number of chunks:", len(chunks))

Number of chunks: 8


In [8]:
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
print("Embedding model loaded")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded


In [9]:
texts = [chunk["text"] for chunk in chunks]
embeddings = embedding_model.encode(texts)
print("Number of embeddings:", len(embeddings))
print("Embedding size:", len(embeddings[0]))

Number of embeddings: 8
Embedding size: 384


In [10]:
import chromadb
client = chromadb.Client()
collection = client.get_or_create_collection(
    name="zepto_policies"
)
print("ChromaDB collection created")

ChromaDB collection created


In [11]:
ids = [chunk["id"] for chunk in chunks]
metadatas = [
    {"document_id": chunk["document_id"]}
    for chunk in chunks
]
collection.add(
    ids=ids,
    documents=texts,
    embeddings=embeddings.tolist(),
    metadatas=metadatas
)
print("Embeddings stored in ChromaDB")
print("Total chunks:", collection.count())

Embeddings stored in ChromaDB
Total chunks: 8


In [12]:
query = "What is the delivery fee?"
query_embedding = embedding_model.encode(query).tolist()
results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3
)
print("Retrieved IDs:")
print(results["ids"][0])
print("\nRetrieved documents:")
for doc in results["documents"][0]:
    print(doc[:200])
    print()

Retrieved IDs:
['doc_01_chunk_01', 'doc_05_chunk_01', 'doc_02_chunk_01']

Retrieved documents:
Zepto delivers grocery and household essentials to serviceable pin codes within 10 to 30 minutes of order confirmation, depending on the customer's delivery zone and current order volume. Standard del

Orders can be cancelled free of cost any time before the order status changes to 'Packed', typically within the first 2 minutes of placing the order. Once an order has been packed, it can no longer be

Grocery and perishable items may be reported for a return within 24 hours of delivery if damaged, spoiled, or incorrect; non-perishable packaged items may be returned within 7 days of delivery in unop



In [13]:
PROMPT_TEMPLATE = """
ROLE:
You are a Zepto policy assistant.

CONTEXT:
Use only the Zepto policy information provided below.

TASK:
Answer the user's question using only the provided context.

FORMAT:
Give a clear and simple answer.

LENGTH:
Answer in 2 to 4 sentences.

NEGATIVE CONSTRAINT:
Do not answer using information that is not present in the provided context.
Do not invent or assume any Zepto policy.

FEW-SHOT EXAMPLE:

Context:
Standard delivery is free on orders over INR 149.

Question:
Is delivery free on orders over INR 149?

Answer:
Yes. Standard delivery is free on orders over INR 149.

NOW ANSWER:

Question:
{question}

Context:
{context}
"""

print(PROMPT_TEMPLATE)


ROLE:
You are a Zepto policy assistant.

CONTEXT:
Use only the Zepto policy information provided below.

TASK:
Answer the user's question using only the provided context.

FORMAT:
Give a clear and simple answer.

LENGTH:
Answer in 2 to 4 sentences.

NEGATIVE CONSTRAINT:
Do not answer using information that is not present in the provided context.
Do not invent or assume any Zepto policy.

FEW-SHOT EXAMPLE:

Context:
Standard delivery is free on orders over INR 149.

Question:
Is delivery free on orders over INR 149?

Answer:
Yes. Standard delivery is free on orders over INR 149.

NOW ANSWER:

Question:
{question}

Context:
{context}



In [14]:
from pydantic import BaseModel, Field
class AnswerResponse(BaseModel):
  answer: str
  sources: list[str]
  confidence: float = Field(
      ge=0,
      le=1
    )

print("Response model created")

Response model created


In [15]:
class QuestionRequest(BaseModel):
    query: str
print("Request model created")

Request model created


In [16]:
from typing import TypedDict
class GraphState(TypedDict):
  query: str
  intent: str
  answer: str
  sources: list[str]
  confidence: float
print("LangGraph state created")

LangGraph state created


In [17]:
import os
MOCK_LLM = os.getenv("MOCK_LLM", "1")
print("MOCK_LLM =", MOCK_LLM)

MOCK_LLM = 1


In [18]:
def classify_intent(state: GraphState):
  query = state["query"].lower()
  keywords = [
        "delivery",
        "return",
        "refund",
        "membership",
        "tracking",
        "cancel",
        "gift card",
        "support hours"
    ]
  if any(keyword in query for keyword in keywords):
    intent = "policy_question"
  else:
    intent = "general_question"
  return {
        "intent": intent
    }

In [19]:
test_state = {
    "query": "What is the delivery fee?",
    "intent": "",
    "answer": "",
    "sources": [],
    "confidence": 0.0
}

print(classify_intent(test_state))

{'intent': 'policy_question'}


In [20]:
test_state["query"] = "What is the capital of India?"
print(classify_intent(test_state))

{'intent': 'general_question'}


In [21]:
def retrieve_and_answer(state: GraphState):

    query = state["query"]

    # Convert query into embedding
    query_embedding = embedding_model.encode(query).tolist()

    # Retrieve top 3 chunks
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=3
    )

    retrieved_documents = results["documents"][0]
    retrieved_ids = results["ids"][0]

    # Most similar chunk
    top_chunk = retrieved_documents[0]

    # MOCK MODE
    if MOCK_LLM != "0":

        answer = (
            "Based on the retrieved context: "
            + top_chunk[:200]
        )

        return {
            "answer": answer,
            "sources": retrieved_ids,
            "confidence": 1.0
        }

    # Optional real LLM branch
    context = "\n\n".join(retrieved_documents)

    prompt = PROMPT_TEMPLATE.format(
        question=query,
        context=context
    )

    answer = "REAL LLM RESPONSE: " + prompt

    return {
        "answer": answer,
        "sources": retrieved_ids,
        "confidence": 1.0
    }

In [22]:
def direct_answer(state: GraphState):
  if MOCK_LLM != "0":
    return {
            "answer": "I can only answer questions about Zepto policies right now.",
            "sources": [],
            "confidence": 1.0
        }
# Optional real LLM branch
  prompt = PROMPT_TEMPLATE.format(
      question=state["query"],
      context=""
    )
  answer = "REAL LLM RESPONSE: " + prompt
  return {
        "answer": answer,
        "sources": [],
        "confidence": 1.0
    }

In [23]:
def route_question(state: GraphState):
  if state["intent"] == "policy_question":
    return "retrieve_and_answer"
  else:
    return "direct_answer"

In [24]:
from langgraph.graph import StateGraph, END

graph = StateGraph(GraphState)

# Add nodes
graph.add_node(
    "classify_intent",
    classify_intent
)

graph.add_node(
    "retrieve_and_answer",
    retrieve_and_answer
)

graph.add_node(
    "direct_answer",
    direct_answer
)

# Starting node
graph.set_entry_point("classify_intent")

# Conditional routing
graph.add_conditional_edges(
    "classify_intent",
    route_question,
    {
        "retrieve_and_answer": "retrieve_and_answer",
        "direct_answer": "direct_answer"
    }
)

# End nodes
graph.add_edge(
    "retrieve_and_answer",
    END
)

graph.add_edge(
    "direct_answer",
    END
)

app_graph = graph.compile()

print("LangGraph created successfully")

LangGraph created successfully


In [25]:
state = {
    "query": "What is the delivery fee?",
    "intent": "",
    "answer": "",
    "sources": [],
    "confidence": 0.0
}
result = app_graph.invoke(state)
print(result)

{'query': 'What is the delivery fee?', 'intent': 'policy_question', 'answer': "Based on the retrieved context: Zepto delivers grocery and household essentials to serviceable pin codes within 10 to 30 minutes of order confirmation, depending on the customer's delivery zone and current order volume. Standard del", 'sources': ['doc_01_chunk_01', 'doc_05_chunk_01', 'doc_02_chunk_01'], 'confidence': 1.0}


In [26]:
state = {
    "query": "What is the capital of India?",
    "intent": "",
    "answer": "",
    "sources": [],
    "confidence": 0.0
}
result = app_graph.invoke(state)
print(result)

{'query': 'What is the capital of India?', 'intent': 'general_question', 'answer': 'I can only answer questions about Zepto policies right now.', 'sources': [], 'confidence': 1.0}


In [27]:
from fastapi import FastAPI
app = FastAPI()

In [28]:
@app.post("/ask", response_model=AnswerResponse)
def ask_question(request: QuestionRequest):
  state = {
        "query": request.query,
        "intent": "",
        "answer": "",
        "sources": [],
        "confidence": 0.0
    }
  result = app_graph.invoke(state)
  response = AnswerResponse(
        answer=result["answer"],
        sources=result["sources"],
        confidence=result["confidence"]
    )
  return response

In [29]:
import subprocess
import time
process = subprocess.Popen(
    [ "uvicorn",
      "main:app",
      "--host",
      "0.0.0.0",
      "--port",
      "8000"
    ]
)
time.sleep(3)
print("FastAPI server started")

FastAPI server started


In [30]:
main_code = '''
import os
from typing import TypedDict

import chromadb
from sentence_transformers import SentenceTransformer
from pydantic import BaseModel, Field
from fastapi import FastAPI
from langgraph.graph import StateGraph, END


# Load documents
documents = []

for i in range(1, 9):

    file_name = f"docs/doc_{i:02d}.txt"

    with open(file_name, "r", encoding="utf-8") as file:
        text = file.read()

    documents.append({
        "id": f"doc_{i:02d}",
        "text": text
    })


# Embedding model
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)


# ChromaDB
client = chromadb.Client()

collection = client.get_or_create_collection(
    name="zepto_policies"
)


# Add documents
if collection.count() == 0:

    for doc in documents:

        embedding = embedding_model.encode(
            doc["text"]
        ).tolist()

        collection.add(
            ids=[doc["id"] + "_chunk_01"],
            documents=[doc["text"]],
            embeddings=[embedding],
            metadatas=[{
                "document_id": doc["id"]
            }]
        )


# Response model
class AnswerResponse(BaseModel):

    answer: str

    sources: list[str]

    confidence: float = Field(
        ge=0,
        le=1
    )


# Request model
class QuestionRequest(BaseModel):

    query: str


# Graph state
class GraphState(TypedDict):

    query: str
    intent: str
    answer: str
    sources: list[str]
    confidence: float


# Mock mode
MOCK_LLM = os.getenv("MOCK_LLM", "1")


# Prompt
PROMPT_TEMPLATE = """
ROLE:
You are a Zepto policy assistant.

CONTEXT:
Use only the Zepto policy information provided below.

TASK:
Answer the user's question using only the provided context.

FORMAT:
Give a clear and simple answer.

LENGTH:
Answer in 2 to 4 sentences.

NEGATIVE CONSTRAINT:
Do not answer using information that is not present in the provided context.

FEW-SHOT EXAMPLE:

Context:
Standard delivery is free on orders over INR 149.

Question:
Is delivery free on orders over INR 149?

Answer:
Yes. Standard delivery is free on orders over INR 149.

Question:
{question}

Context:
{context}
"""


# Node 1
def classify_intent(state):

    query = state["query"].lower()

    keywords = [
        "delivery",
        "return",
        "refund",
        "membership",
        "tracking",
        "cancel",
        "gift card",
        "support hours"
    ]

    if any(word in query for word in keywords):

        return {
            "intent": "policy_question"
        }

    return {
        "intent": "general_question"
    }


# Node 2
def retrieve_and_answer(state):

    query = state["query"]

    query_embedding = embedding_model.encode(
        query
    ).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=3
    )

    docs = results["documents"][0]
    ids = results["ids"][0]

    top_chunk = docs[0]

    if MOCK_LLM != "0":

        return {
            "answer":
                "Based on the retrieved context: "
                + top_chunk[:200],

            "sources": ids,

            "confidence": 1.0
        }

    context = "\\n\\n".join(docs)

    prompt = PROMPT_TEMPLATE.format(
        question=query,
        context=context
    )

    return {
        "answer": "REAL LLM RESPONSE: " + prompt,
        "sources": ids,
        "confidence": 1.0
    }


# Node 3
def direct_answer(state):

    if MOCK_LLM != "0":

        return {
            "answer":
                "I can only answer questions about Zepto policies right now.",

            "sources": [],

            "confidence": 1.0
        }

    prompt = PROMPT_TEMPLATE.format(
        question=state["query"],
        context=""
    )

    return {
        "answer": "REAL LLM RESPONSE: " + prompt,
        "sources": [],
        "confidence": 1.0
    }


# Router
def route_question(state):

    if state["intent"] == "policy_question":

        return "retrieve_and_answer"

    return "direct_answer"


# LangGraph
graph = StateGraph(GraphState)

graph.add_node(
    "classify_intent",
    classify_intent
)

graph.add_node(
    "retrieve_and_answer",
    retrieve_and_answer
)

graph.add_node(
    "direct_answer",
    direct_answer
)

graph.set_entry_point(
    "classify_intent"
)

graph.add_conditional_edges(
    "classify_intent",
    route_question,
    {
        "retrieve_and_answer": "retrieve_and_answer",
        "direct_answer": "direct_answer"
    }
)

graph.add_edge(
    "retrieve_and_answer",
    END
)

graph.add_edge(
    "direct_answer",
    END
)

app_graph = graph.compile()


# FastAPI
app = FastAPI()


@app.post("/ask", response_model=AnswerResponse)
def ask_question(request: QuestionRequest):

    state = {
        "query": request.query,
        "intent": "",
        "answer": "",
        "sources": [],
        "confidence": 0.0
    }

    result = app_graph.invoke(state)

    return AnswerResponse(
        answer=result["answer"],
        sources=result["sources"],
        confidence=result["confidence"]
    )
'''

with open("main.py", "w", encoding="utf-8") as file:
    file.write(main_code)

print("main.py created")

main.py created


In [31]:
requirements = """fastapi
uvicorn
chromadb
sentence-transformers
langgraph
pydantic
"""

with open("requirements.txt", "w") as file:
    file.write(requirements)

print("requirements.txt created")

requirements.txt created


In [32]:
dockerfile = """FROM python:3.11

WORKDIR /app

COPY requirements.txt .

RUN pip install --no-cache-dir -r requirements.txt

COPY . .

EXPOSE 7860

CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "7860"]
"""

with open("Dockerfile", "w") as file:
    file.write(dockerfile)

print("Dockerfile created")

Dockerfile created


In [33]:
readme = """# Zepto Policy RAG

## Architecture

The project follows this RAG pipeline:

Documents
    |
    v
Ingestion
    |
    v
Chunking
    |
    v
all-MiniLM-L6-v2
    |
    v
ChromaDB
    |
    v
LangGraph
    |
    v
Intent Classification
    |
    +-----------------------+
    |                       |
    v                       v
Policy Question       General Question
    |                       |
    v                       v
Retrieve Top 3        Direct Answer
    |
    v
Final Answer
    |
    v
Pydantic JSON Response

## Ingestion

The eight documents are stored in the docs folder.

The documents are loaded by main.py.

Because the documents are short, each document is treated as one chunk.

## Embedding

The all-MiniLM-L6-v2 model creates embeddings for every chunk.

## ChromaDB

The embeddings are stored in the ChromaDB collection:

zepto_policies

## Retrieval

The retrieve_and_answer node embeds the user's query and retrieves the top 3 most similar chunks from ChromaDB.

## Generation

In the default MOCK_LLM mode, no real LLM is called.

The answer is generated using the most similar retrieved chunk.

For general questions, the direct_answer node returns a fixed response.

## MOCK_LLM

The default value is:

MOCK_LLM=1

In mock mode:

- Intent classification uses keyword matching.
- Retrieval still uses real embeddings and ChromaDB.
- The final answer is generated by deterministic code.
- General questions receive a fixed answer.

When MOCK_LLM=0, the optional real LLM branch can be connected.

## Structured Prompt

The prompt contains:

- Role
- Context
- Task
- Format
- Length
- Negative constraint
- Few-shot example

The negative constraint says:

Do not answer using information that is not present in the provided context.

## API

POST /ask

Example request:

{
    "query": "What is the delivery fee?"
}

Example policy response:

{
    "answer": "Based on the retrieved context: ...",
    "sources": ["doc_01_chunk_01"],
    "confidence": 1.0
}

Example general question:

{
    "query": "What is the capital of India?"
}

Response:

{
    "answer": "I can only answer questions about Zepto policies right now.",
    "sources": [],
    "confidence": 1.0
}

## Run locally

pip install -r requirements.txt

uvicorn main:app --reload

## Docker

docker build -t zepto-rag .

docker run -p 7860:7860 zepto-rag
"""

with open("README.md", "w") as file:
    file.write(readme)

print("README.md created")

README.md created


In [34]:
import os
print("Project files:")
for root, dirs, files in os.walk("."):
  for file in files:
    if (
        file.endswith(".txt")
        or file.endswith(".py")
        or file.endswith(".md")
        or file == "Dockerfile"
        ):

          print(os.path.join(root, file))

Project files:
./Dockerfile
./main.py
./requirements.txt
./README.md
./docs/doc_02.txt
./docs/doc_03.txt
./docs/doc_04.txt
./docs/doc_08.txt
./docs/doc_07.txt
./docs/doc_05.txt
./docs/doc_06.txt
./docs/doc_01.txt
./sample_data/README.md


In [36]:
!python -c "import main; print('main.py loaded successfully')"

Loading weights: 100% 103/103 [00:00<00:00, 4385.61it/s]
main.py loaded successfully


In [37]:
import subprocess
import time
server = subprocess.Popen(
    [ "uvicorn",
      "main:app",
      "--host",
      "0.0.0.0",
      "--port",
      "8000"
    ]
)
time.sleep(5)
print("FastAPI server is running")

FastAPI server is running


In [41]:
import requests
response = requests.post(
    "http://127.0.0.1:8000/ask",
    json={
        "query": "What is the delivery fee?"
    }
)
print(response.status_code)
print(response.json())

200
{'answer': "Based on the retrieved context: Zepto delivers grocery and household essentials to serviceable pin codes within 10 to 30 minutes of order confirmation, depending on the customer's delivery zone and current order volume. Standard del", 'sources': ['doc_01_chunk_01', 'doc_05_chunk_01', 'doc_02_chunk_01'], 'confidence': 1.0}


In [42]:
response = requests.post(
    "http://127.0.0.1:8000/ask",
    json={
        "query": "What is the capital of India?"
    }
)

print(response.status_code)
print(response.json())

200
{'answer': 'I can only answer questions about Zepto policies right now.', 'sources': [], 'confidence': 1.0}
